## 임베딩

In [1]:
# .env에 저장된 API Key 등 환경변수 로드
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from langchain_openai import OpenAIEmbeddings

# 텍스트를 숫자 벡터로 변환할 임베딩 모델 생성
# dimensions는 출력 벡터의 차원 수를 지정한다.
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024
)

text = "임베딩 테스트를 하기 위한 샘플 문장."

# 단일 검색어/질문 임베딩
# 결과는 [0.01, -0.02, ...] 형태의 실수 벡터
query_result = embeddings.embed_query(text)

print(len(query_result))   # 벡터 차원 확인
print(query_result[:5])    # 앞의 일부 값만 확인

1024
[-0.00949859619140625, 0.04168701171875, 0.0235595703125, -0.024566650390625, 0.01751708984375]


In [7]:
# 여러 문서를 한 번에 임베딩
# RAG에서는 보통 저장할 Chunk들을 embed_documents()로 벡터화한다.
doc_result = embeddings.embed_documents(
    [text, text, text, text]
)

# 첫 번째 문서 벡터의 일부 확인
doc_result[0][:5]

[-0.0095367431640625,
 0.0416259765625,
 0.023529052734375,
 -0.02459716796875,
 0.0174713134765625]

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# 의미가 비슷하거나 다른 문장들을 준비해 임베딩 결과를 비교
sentence1 = "안녕하세요? 반갑습니다."
sentence2 = "안녕하세요? 반갑습니다!"
sentence3 = "안녕하세요? 만나서 반가워요."
sentence4 = "Hi, nice to meet you."
sentence5 = "I like to eat apples"

sentences = [sentence1, sentence2, sentence3, sentence4, sentence5]

# 같은 임베딩 모델로 모든 문장을 같은 벡터 공간에 배치
embedded_sentences = embeddings.embed_documents(sentences)

In [10]:
def similarity(a, b):
    # 코사인 유사도: 두 벡터의 방향이 얼마나 비슷한지 계산
    # 값이 클수록 두 문장의 의미가 가깝다고 해석할 수 있다.
    return cosine_similarity([a], [b])[0][0]

In [11]:
# 모든 문장 쌍의 코사인 유사도 비교
# i < j 조건으로 같은 쌍을 중복 출력하지 않는다.
for i, sentence in enumerate(embedded_sentences):
    for j, other_sentence in enumerate(embedded_sentences):
        if i < j:
            print(
                f"[유사도 {similarity(sentence, other_sentence):.4f}] "
                f"{sentences[i]} \t <====> \t {sentences[j]}"
            )

[유사도 0.9644] 안녕하세요? 반갑습니다. 	 <====> 	 안녕하세요? 반갑습니다!
[유사도 0.8422] 안녕하세요? 반갑습니다. 	 <====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.5043] 안녕하세요? 반갑습니다. 	 <====> 	 Hi, nice to meet you.
[유사도 0.1295] 안녕하세요? 반갑습니다. 	 <====> 	 I like to eat apples
[유사도 0.8185] 안녕하세요? 반갑습니다! 	 <====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.4791] 안녕하세요? 반갑습니다! 	 <====> 	 Hi, nice to meet you.
[유사도 0.1343] 안녕하세요? 반갑습니다! 	 <====> 	 I like to eat apples
[유사도 0.5164] 안녕하세요? 만나서 반가워요. 	 <====> 	 Hi, nice to meet you.
[유사도 0.1337] 안녕하세요? 만나서 반가워요. 	 <====> 	 I like to eat apples
[유사도 0.2061] Hi, nice to meet you. 	 <====> 	 I like to eat apples
